## **TESTING SCRIPTS**

### **Test parameters**
Number of floorplans ``N = 10``

#### **``run_subset.py``**

**Total runtime: 89.647 seconds**

In [8]:
total_time = 4571/10 * 89.647

print(f"Expected time to run on all data: ", total_time/60, "mins\n", 
"                                 ", total_time/360, "hours")

Expected time to run on all data:  682.9607283333335 mins
                                   113.82678805555558 hours


Profiling shows that the main bottleneck is the Jacobi update expression. 89.5128s of 89.647s are spent on the ``jacobi()`` function.

In particular, line 38 accounts for about 50.3% of the measured time, where the left-neighbor slice participates in the vectorized sum. However, this does not mean that only the left-neighbor access is inherently expensive by itself; rather, line profiling attributes much of the cost of the full multi-array expression to that line.

Significant time is also spent on lines 44–45, where masked indexing extracts the current and newly computed values at interior points, on line 47 where the maximum update difference is computed, and on line 48 where masked assignment writes values back into the grid.

The underlying issue is not that every array access duplicates the whole array. Ordinary slicing such as u[1:-1, :-2] usually creates a view, not a full copy. The expensive part is that the vectorized arithmetic and masked indexing create temporary arrays and involve substantial memory traffic. In particular, boolean indexing like ``u[1:-1, 1:-1][interior_mask]`` produces a copy of the selected elements, which is more memory-intensive and less cache-friendly.

___

#### **``run_subset_2.py``**


**Total runtime: 13.758 seconds**

In [10]:
total_time = 4571/10 * 13.758

print(f"Expected time to run on all data: ", total_time/60, "mins\n", 
"                                 ", total_time/360, "hours")

Expected time to run on all data:  104.81303 mins
                                   17.468838333333334 hours
